In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from matplotlib.animation import FuncAnimation

<h2> Nokken-ontwerp: specificaties </h2>
Gegevens : De te ontwerpen nok moet volgende heffing kunnen realiseren :

1. van 45° tot 90° : +5 mm
2. van 90° tot 180° : +15 mm
3. van 180° tot 305° : -20 mm

De equivalente massa en dempingsconstante van de volger (en bijhorende onderdelen) worden respectievelijk geschat op 15 kg en 0.055, terwijl het mechanisme volgende statische krachten moet uitoefenen:

1. van 45° tot 90° : een konstante druk-kracht van 250 N
2. van 90° tot 180° : een lineair toenemende druk-kracht van 250 N tot 1050 N
3. van 180° tot 250° : een konstante trek-kracht van 300 N

De gevraagde cyclustijd voor de bewerking uitgevoerd door de volger is 0.3 sec.

Hieronder tonen we de gegeven specs in een grafiek van de positie en de kracht

In [ ]:
# ---------------------------------------------------------
# Gegeven specificaties
# ---------------------------------------------------------

theta = np.linspace(0, 360, 1000)   # hoek van de nok in graden

# Krachtwaarden
F_pre = 250      # N, constante drukkracht tussen 45° en 90°
F_max = 1050     # N, maximale drukkracht bij 180°
F_pull = -300    # N, trekkracht

# Let op:
# In de opdracht staat trekkracht van 180° tot 250°.
# Als je de trekkracht tot 305° wilt tonen, zet deze waarde op 305.
theta_pull_end = 250


# ---------------------------------------------------------
# Positieprofiel y(theta)
# ---------------------------------------------------------

y = np.zeros_like(theta)

# 45° tot 90°: van 0 mm naar 5 mm
mask = (theta >= 45) & (theta < 90)
y[mask] = (theta[mask] - 45) / (90 - 45) * 5

# 90° tot 180°: van 5 mm naar 20 mm
mask = (theta >= 90) & (theta < 180)
y[mask] = 5 + (theta[mask] - 90) / (180 - 90) * 15

# 180° tot 305°: van 20 mm naar 0 mm
mask = (theta >= 180) & (theta < 305)
y[mask] = 20 - (theta[mask] - 180) / (305 - 180) * 20

# 0°-45° en 305°-360° blijven 0 mm


# ---------------------------------------------------------
# Externe kracht F(theta)
# ---------------------------------------------------------

F = np.zeros_like(theta)

# 45° tot 90°: constante drukkracht
mask = (theta >= 45) & (theta < 90)
F[mask] = F_pre

# 90° tot 180°: lineair stijgend van F_pre naar F_max
mask = (theta >= 90) & (theta < 180)
F[mask] = F_pre + (theta[mask] - 90) / (180 - 90) * (F_max - F_pre)

# 180° tot theta_pull_end: constante trekkracht
mask = (theta >= 180) & (theta < theta_pull_end)
F[mask] = F_pull

# Na theta_pull_end blijft de kracht 0 N


# ---------------------------------------------------------
# Grafieken
# ---------------------------------------------------------

fig, axs = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Grafiek 1: positie
axs[0].plot(theta, y, linewidth=2)
axs[0].set_ylabel("y-positie volger (mm)")
axs[0].set_title("Gegeven positieprofiel van de volger")
axs[0].set_xlim(0, 360)
axs[0].set_ylim(0, 25)
axs[0].set_yticks(np.arange(0, 26, 5))
axs[0].grid(True)

# Grafiek 2: kracht
axs[1].plot(theta, F, linewidth=2)
axs[1].set_xlabel("Hoek van de nok (°)")
axs[1].set_ylabel("Kracht op volger (N)")
axs[1].set_title("Gegeven extern krachtprofiel op de volger")
axs[1].set_xlim(0, 360)
axs[1].set_ylim(-350, 1100)
axs[1].set_xticks(np.arange(0, 361, 30))
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Toepassing: Tablet Press

Op basis van het krachtenprofiel dachten we een soort stempelmachine te maken. Omdat de krachten niet ongelofelijk hoog zijn was de toepassing tablet press voor de farmaseutische industrie boven gekomen. Hieronder is een schets van de verschillende stappen met daaronder uitleg.

<p align="center">
  <img src="Images\visualisatie_compressie.png" width="700"><br>
  <b>Figuur 1:</b> Visualie compressiestappen
</p>

### Positie 1: Rust
**Hoeken: 305°-45°**

De stempel blijft zijn positie houden en ondervind geen externe krachten. Tijdens deze fase wordt de afgemaakte pil afgevoerd en een nieuwe matrijs gevuld met poeder klaargezet voor de volgende compressie.

### Positie 2: Plaatsing stempel
**Hoek: $\theta_1$**

De stempel in rust is 2.5mm verwijderd van de opening van de matrijs. Van positie 1 naar positie 2 verplaatst de stempel dus +2.5mm zonder kracht te ondervinden. De reden dat deze hoek $\theta_1$ wordt genoemd is omdat uiteindelijk het verplaatsingsprofiel wordt verzacht om minder sterke schokken te krijgen (deze moet dus berekend worden).

### Positie 3: Indrukken stempel
**Hoek: 90°**

De stempel positioneert zich 2.5mm in de matrijs (afmaken van 5mm verplaatsing). Binnen de matrijs in de neerwaarste slag zal de stempel een constante tegenwerkende wrijving van 250N ondervinden (daarom dat van 1 $\rightarrow$ 3 opgesplitst moest worden). De kleine compressie van de poeder die hier gebeurt wordt verwaarloosd.

### Positie 4: Compressie + Dwell
**Hoeken: 150°-180°**

Van 90°-150° is de compressieslag van de stempel waar de stempel 15mm in de matrijs wordt geduwd. Compressie wordt gemodelleerd als evenredig met de samendrukking en dus lineair toenemend tot 1050N. Hierbovenop wordt de wrijving in de matrijs gesuperponeerd.

Van 150°-180° blijft de stempel op dezelfde positie. Dit heet dwelling en zorgt ervoor dat er minder kans is op barsten in de pil. In deze positie ondervind de stempel altijd dezelfde kracht van 1050N.

### Positie 5: Uittrekken
**Hoek: $\theta_2$**

Van 180°-$\theta_2$ wordt de stempel 17.5mm uitgetrokken. Op deze positie is de stempel net uit de matrijs en tot dan ondervindt het een tegenwerkende kracht van -300N.

### Positie 6: Terug naar startpositie
**Hoek: 305°**

De stempel verplaatst de laatste 2.5mm terug naar de startpositie. Tijdens deze beweging ondervindt de stempel geen uitwendige kracht.


## Keuze van de bewegingswetten

Voor de vrije verplaatsing en de eerste intrede in de matrijs wordt een cycloïdale bewegingswet gebruikt. Deze bewegingswet heeft een nulwaarde voor snelheid en versnelling aan het begin en einde van het segment. Daardoor wordt de overgang tussen rust en beweging vloeiender dan bij een lineair profiel. Dit is belangrijk omdat plotse versnellingen trillingen kunnen veroorzaken, wat nadelig is voor een gelijkmatige poederverdeling in de matrijs.

Voor de hoofdcompressie wordt een 7de-graads polynoom gebruikt. Dit segment is het meest kritisch, omdat hier de grootste krachten optreden en de tablet effectief gevormd wordt. De 7de-graads polynoom zorgt ervoor dat positie, snelheid, versnelling en jerk op een gecontroleerde manier naar nul kunnen gaan aan de grenzen van het segment. Hierdoor worden schokken en abrupte veranderingen in de dynamische belasting beperkt.

Tussen 150° en 180° wordt een dwell voorzien. Tijdens deze fase blijft de stempel op maximale indrukking staan. Dit laat toe om de maximale compressiekracht gedurende een korte tijd aan te houden, wat de kwaliteit en vormvastheid van de tablet kan verbeteren.

Voor de terugtrekkende beweging wordt opnieuw een cycloïdale bewegingswet gebruikt. Deze beweging is minder kritisch dan de hoofdcompressie, maar moet nog steeds voldoende vloeiend verlopen om beschadiging van de gevormde tablet te vermijden. De laatste verplaatsing terug naar de startpositie gebeurt opnieuw zonder externe kracht.

In [ ]:
# ---------------------------------------------------------
# Specificaties toepassing: tablet press
# ---------------------------------------------------------

dtheta = 0.01          # resolutie in graden
T = 0.3                # cyclustijd in s
omega = 2 * np.pi / T  # hoeksnelheid in rad/s

theta_deg = np.arange(0, 360, dtheta)   # hoek in graden
theta = theta_deg * np.pi / 180         # hoek in radialen

lift = np.zeros_like(theta_deg)         # y-positie [mm]
vel_deg = np.zeros_like(theta_deg)      # dy/dtheta [mm/deg]
acc_deg = np.zeros_like(theta_deg)      # d²y/dtheta² [mm/deg²]
jerk_deg = np.zeros_like(theta_deg)     # d³y/dtheta³ [mm/deg³]
ext_load = np.zeros_like(theta_deg)     # externe kracht [N]
pressure_angle = np.zeros_like(theta_deg)


# ---------------------------------------------------------
# Bewegingswetten
# ---------------------------------------------------------

def motion_law(u, law):
    """
    Geeft dimensieloze positie q, snelheid dq, versnelling ddq en jerk dddq
    als functie van u = (theta - theta_start) / beta.
    De afgeleiden zijn naar u, niet naar theta.
    """

    if law == 1:  # dwell
        q = np.zeros_like(u)
        dq = np.zeros_like(u)
        ddq = np.zeros_like(u)
        dddq = np.zeros_like(u)

    elif law == 6:  # cycloïde
        q = u - np.sin(2 * np.pi * u) / (2 * np.pi)
        dq = 1 - np.cos(2 * np.pi * u)
        ddq = 2 * np.pi * np.sin(2 * np.pi * u)
        dddq = 4 * np.pi**2 * np.cos(2 * np.pi * u)

    elif law == 7:  # 5de-graads polynoom
        q = 10*u**3 - 15*u**4 + 6*u**5
        dq = 30*u**2 - 60*u**3 + 30*u**4
        ddq = 60*u - 180*u**2 + 120*u**3
        dddq = 60 - 360*u + 360*u**2

    elif law == 9:  # 7de-graads polynoom
        q = 35*u**4 - 84*u**5 + 70*u**6 - 20*u**7
        dq = 140*u**3 - 420*u**4 + 420*u**5 - 140*u**6
        ddq = 420*u**2 - 1680*u**3 + 2100*u**4 - 840*u**5
        dddq = 840*u - 5040*u**2 + 8400*u**3 - 4200*u**4

    else:
        raise ValueError(f"Onbekende bewegingswet: {law}")

    return q, dq, ddq, dddq


def add_motion_segment(startangle, endangle, startlift, endlift, motionlaw):
    """
    Vult lift, vel_deg, acc_deg en jerk_deg voor één bewegingssegment.
    De globale variabelen blijven zo bruikbaar in de rest van de notebook.
    """

    assert endangle > startangle, "End angle should be bigger than start angle"

    mask = (theta_deg >= startangle) & (theta_deg < endangle)

    beta = endangle - startangle
    L = endlift - startlift

    u = (theta_deg[mask] - startangle) / beta
    q, dq, ddq, dddq = motion_law(u, motionlaw)

    lift[mask] = startlift + L * q
    vel_deg[mask] = L / beta * dq
    acc_deg[mask] = L / beta**2 * ddq
    jerk_deg[mask] = L / beta**3 * dddq


def find_theta_at_lift(startangle, endangle, target_lift):
    """
    Zoekt de hoek waarbij de lift in een segment target_lift bereikt.
    Werkt zowel voor stijgende als dalende beweging.
    """

    mask = (theta_deg >= startangle) & (theta_deg < endangle)

    th = theta_deg[mask]
    y = lift[mask]

    crossings = np.where((y[:-1] - target_lift) * (y[1:] - target_lift) <= 0)[0]

    if len(crossings) == 0:
        raise ValueError(
            f"Lift {target_lift} mm wordt niet bereikt tussen {startangle}° en {endangle}°"
        )

    i = crossings[0]

    # lineaire interpolatie tussen twee dichtstbijzijnde punten
    theta_crossing = th[i] + (target_lift - y[i]) * (th[i+1] - th[i]) / (y[i+1] - y[i])

    return theta_crossing

In [ ]:
# ---------------------------------------------------------
# Segmenten van het aangepaste positieprofiel
# ---------------------------------------------------------

startangle1 = 0
endangle1 = 45
theta_seg1 = endangle1 - startangle1
startlift1 = 0
endlift1 = 0
motionlaw1 = 1      # dwell


startangle2 = 45
endangle2 = 90
theta_seg2 = endangle2 - startangle2
startlift2 = 0
endlift2 = 5
motionlaw2 = 6      # cycloïde


startangle3 = 90
endangle3 = 150
theta_seg3 = endangle3 - startangle3
startlift3 = 5
endlift3 = 20
motionlaw3 = 9      # 7de-graads polynoom


startangle4 = 150
endangle4 = 180
theta_seg4 = endangle4 - startangle4
startlift4 = 20
endlift4 = 20
motionlaw4 = 1      # dwell


startangle5 = 180
endangle5 = 305
theta_seg5 = endangle5 - startangle5
startlift5 = 20
endlift5 = 0
motionlaw5 = 6      # cycloïde


startangle6 = 305
endangle6 = 360
theta_seg6 = endangle6 - startangle6
startlift6 = 0
endlift6 = 0
motionlaw6 = 1      # dwell


motion_segments = [
    (startangle1, endangle1, startlift1, endlift1, motionlaw1),
    (startangle2, endangle2, startlift2, endlift2, motionlaw2),
    (startangle3, endangle3, startlift3, endlift3, motionlaw3),
    (startangle4, endangle4, startlift4, endlift4, motionlaw4),
    (startangle5, endangle5, startlift5, endlift5, motionlaw5),
    (startangle6, endangle6, startlift6, endlift6, motionlaw6),
]

for segment in motion_segments:
    add_motion_segment(*segment)


# Oude conventie uit jullie notebook: afgeleiden naar radialen
vel = vel_deg * 180 / np.pi
acc = acc_deg * (180 / np.pi)**2
jerk = jerk_deg * (180 / np.pi)**3

# Extra: echte tijdsafgeleiden, handig voor interpretatie en dynamica
vel_time = vel * omega
acc_time = acc * omega**2
jerk_time = jerk * omega**3

In [ ]:
# ---------------------------------------------------------
# Bepaal theta1 en theta2 uit het gesmoothde positieprofiel
# ---------------------------------------------------------

# theta1: stempel raakt de opening van de matrijs bij y = 2.5 mm
theta1 = find_theta_at_lift(45, 90, 2.5)

# theta2: stempel is opnieuw net uit de matrijs bij y = 2.5 mm
theta2 = find_theta_at_lift(180, 305, 2.5)

print(f"theta1 = {theta1:.2f}°")
print(f"theta2 = {theta2:.2f}°")

In [ ]:
# ---------------------------------------------------------
# Aangepast krachtenprofiel
# ---------------------------------------------------------

def add_load_segment(startangle, endangle, startload, endload):
    """
    Vult ext_load voor één krachtsegment.
    Kracht verloopt lineair van startload naar endload.
    """

    if np.isclose(startangle, endangle):
        return

    mask = (theta_deg >= startangle) & (theta_deg < endangle)

    u = (theta_deg[mask] - startangle) / (endangle - startangle)
    ext_load[mask] = startload + u * (endload - startload)


# Van 45° tot theta1: geen externe kracht,
# want de stempel beweegt nog vrij naar de opening van de matrijs.

startangleload1 = theta1
endangleload1 = 90
startload1 = 250
endload1 = 250


startangleload2 = 90
endangleload2 = 150
startload2 = 250
endload2 = 1050


startangleload3 = 150
endangleload3 = 180
startload3 = 1050
endload3 = 1050


startangleload4 = 180
endangleload4 = theta2
startload4 = -300
endload4 = -300


load_segments = [
    (startangleload1, endangleload1, startload1, endload1),
    (startangleload2, endangleload2, startload2, endload2),
    (startangleload3, endangleload3, startload3, endload3),
    (startangleload4, endangleload4, startload4, endload4),
]

for segment in load_segments:
    add_load_segment(*segment)

In [ ]:
# ---------------------------------------------------------
# Gealigneerde grafieken: positie, snelheid en kracht
# ---------------------------------------------------------

fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

key_angles = [45, theta1, 90, 150, 180, theta2, 305]

# Positie
axs[0].plot(theta_deg, lift, linewidth=2)
axs[0].set_ylabel("y-positie [mm]")
axs[0].set_title("Aangepast positieprofiel van de stempel")
axs[0].set_ylim(0, 25)
axs[0].set_yticks(np.arange(0, 26, 5))
axs[0].grid(True)

# Kracht
axs[1].plot(theta_deg, ext_load, linewidth=2)
axs[1].set_ylabel("externe kracht [N]")
axs[1].set_xlabel("hoek van de nok [°]")
axs[1].set_title("Aangepast extern krachtenprofiel")
axs[1].set_ylim(-350, 1100)
axs[1].set_yticks([-300, 0, 250, 1050])
axs[1].grid(True)

# Gemeenschappelijke x-as en verticale hulplijnen
for ax in axs:
    ax.set_xlim(0, 360)
    ax.set_xticks(np.arange(0, 361, 30))

    for angle in key_angles:
        ax.axvline(angle, linestyle="--", linewidth=0.8, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# compute motion functions:

# a motion segment runs in one periodic cycle
# from "startangle" till "endangle",
# and moves from "startlift" till "endlift",
# with a given "motionlaw", and
# with interpolation resolution "dtheta":

def addMotionSegment(startangle,endangle,startlift,endlift,motionlaw,dtheta,T):
    assert startangle - endangle < 0, 'End angle should be bigger than start angle'
    start_index = int(startangle/dtheta)
    end_index = int(endangle/dtheta)
    beta = endangle - startangle
    L0 = startlift
    L1 = endlift
    x = np.linspace(0, 1, (end_index - start_index))
    omega = 2*np.pi / T
    if motionlaw == 1:  # dwell
        assert L1 - L0 == 0, 'The given input does not represent a dwell'
        lift[start_index:end_index] = L0 * np.ones_like(x)
        vel_deg[start_index:end_index] = np.zeros_like(x)
        acc_deg[start_index:end_index] = np.zeros_like(x)
        jerk_deg[start_index:end_index] = np.zeros_like(x)

    elif motionlaw == 2:  # minimal acceleration / Bang-bang versnelling
        L = L1 - L0
        n = end_index - start_index
        mid = n // 2

        # lift
        lift[start_index:start_index+mid] = L * 2 * x[:mid]**2 + L0
        lift[start_index+mid:end_index] = L * (-2*x[mid:]**2 + 4*x[mid:] - 1) + L0

        # velocity
        vel_deg[start_index:start_index+mid] =  L / beta * 4 * x[:mid]
        vel_deg[start_index+mid:end_index] =  L / beta * (-4*x[mid:] + 4)

        # acceleration
        acc_deg[start_index:start_index+mid] =  L / beta**2 * 4
        acc_deg[start_index+mid:end_index] = L / beta**2 * (-4)

        jerk_deg[start_index:start_index+mid] = np.zeros_like(x[:mid])
        jerk_deg[start_index+mid:end_index] = np.zeros_like(x[mid:])

    elif motionlaw == 3:  # 3rd order polynomial (minimal rms acceleration)
        L = L1 - L0
        lift[start_index:end_index] =  L * (3 * x**2 - 2 * x**3) + L0
        vel_deg[start_index:end_index] = L / beta * (6 * x - 6 * x**2)
        acc_deg[start_index:end_index] =  L / beta**2 * (6 - 12 * x)
        jerk_deg[start_index:end_index] = L / beta**3 * (-12)

    elif motionlaw == 4:  # harmonische
        L = L1 - L0
        lift[start_index:end_index] = L * (1 - np.cos(np.pi * x)) / 2 + L0
        vel_deg[start_index:end_index] =  L / beta * np.sin(np.pi * x) * np.pi / 2
        acc_deg[start_index:end_index] =  L / beta**2 * np.cos(np.pi * x) * np.pi**2 / 2
        jerk_deg[start_index:end_index] = np.zeros_like(x)

    elif motionlaw == 5:  # minimale ruk
        L = L1 - L0
        lift[start_index: 0.25*len(x)] = L * 16*x**3 / 3 + L0
        lift[0.25*len(x): 0.75*len(x)] = L * -16*x**3 / 3 + 8*x**2 - 2*x + 1/6 + L0
        lift[0.75*len(x): end_index] = L * 16*x**3 / 3 - 16*x**2 + 16*x/3 - 13/3 + L0
        vel_deg[start_index: 0.25*len(x)] =  L / beta * 16*x**2
        vel_deg[0.25*len(x): 0.75*len(x)] =  L / beta * (-16*x**2  + 16*x - 2)
        vel_deg[0.75*len(x): end_index] =  L / beta * (16*x**2 - 32*x + 16)
        acc_deg[start_index: 0.25*len(x)] =  L / beta**2 * 32*x
        acc_deg[0.25*len(x): 0.75*len(x)] =  L / beta**2 * (-32*x + 16)
        acc_deg[0.75*len(x): end_index] =  L / beta**2 * (32*x - 32)
        jerk_deg[start_index: 0.25*len(x)] =  L / beta**3 * 32
        jerk_deg[0.25*len(x): 0.75*len(x)] =  L / beta**3 * (-32)
        jerk_deg[0.75*len(x): end_index] =  L / beta**3 * 32


    elif motionlaw == 6:  # volle cycloide
        L = L1 - L0
        lift[start_index:end_index] = L * (x - np.sin(2 * np.pi * x) / (2 * np.pi)) + L0
        vel_deg[start_index:end_index] =  L / beta * (1 - np.cos(2 * np.pi * x))
        acc_deg[start_index:end_index] =  2 * np.pi * L / beta**2 * np.sin(2 * np.pi * x)
        jerk_deg[start_index:end_index] =  4 * np.pi**2 * L / beta**3 * np.cos(2 * np.pi * x)

    elif motionlaw == 7:  # 5th degree poly
        L = L1 - L0
        lift[start_index:end_index] = L0 + L * (6 * x**5 - 15 * x**4 + 10 * x**3)
        vel_deg[start_index:end_index] = L / beta * (30 * x**4 - 60 * x**3 + 30 * x**2)
        acc_deg[start_index:end_index] = L / beta**2 * (120 * x**3 - 180 * x**2 + 60 * x)
        jerk_deg[start_index:end_index] =  L / beta**3 * (360 * x**2 - 360 * x + 60)

    elif motionlaw == 8:  # minimale afgeleide van de ruk
        L = L1 - L0
        i = (1 - np.sqrt(0.5))
        lift[start_index:i*len(x)] = L * 16*x**4 + L0
        lift[i*len(x): 0.5*len(x)] = L * (-16*x**4 + 128*i*x**3 - 192*i**2*x**2 + 128*i**3*x - 32*i**4) + L0
        lift[0.5*len(x): (1-i)*len(x)] = L * (1 + 16*(1 - x)**4 - 128*i*(1 - x)**3 
                                              + 192*i**2*(1 - x)**2 - 128*(1-x)*i**3 + 32*i**4) + L0
        lift[(1-i)*len(x): end_index] = L * (1 - 16*(1-x)**4) + L0
        vel_deg[start_index:i*len(x)] = L / beta * 64*x**3
        vel_deg[i*len(x): 0.5*len(x)] =  L / beta * (-64*x**3 + 384*i*x**2 - 384*i**2*x + 128*i**3)
        vel_deg[0.5*len(x): (1-i)*len(x)] =  L / beta * (-64*(1-x)**3 + 384*i*(1-x)**2 - 384*i**2*(1-x) + 128*i**3)
        vel_deg[(1-i)*len(x): end_index] = L / beta * 64*(1-x)**3
        acc_deg[start_index:i*len(x)] =  L / beta**2 * 192*x**2
        acc_deg[i*len(x): 0.5*len(x)] = L / beta**2 * (-192*x**2 + 768*i*x - 384*i**2)
        acc_deg[0.5*len(x): (1-i)*len(x)] =  L / beta**2 * (192*(1-x)**2 - 768*i*(1-x) + 384*i**2)
        acc_deg[(1-i)*len(x): end_index] =  L / beta**2 * (-192*(1-x)**2)

    elif motionlaw == 9:  # 7th degree poly
        L = L1 - L0
        lift[start_index:end_index] = L0 + L * (-20 * x**7 + 70 * x**6 - 84 * x**5 + 35 * x**4)
        vel_deg[start_index:end_index] =L / beta * (-140 * x**6 + 420 * x**5 - 420 * x**4 + 140 * x**3)
        acc_deg[start_index:end_index] = L / beta**2 * (-840 * x**5 + 2100 * x**4 - 1680 * x**3 + 420 * x**2)
        jerk_deg[start_index:end_index] = L / beta**3 * (-4200 * x**4 + 8400 * x**3 - 5040 * x**2 + 840 * x)

    vel = vel_deg*180/np.pi
    acc = acc_deg*(180/np.pi)**2
    jerk = jerk_deg*(180/np.pi)**3

    return lift, vel_deg, acc_deg, vel, acc, jerk



def plotMotionLaw(t,lift,vel,acc,jerk):
    fig1, ax1 = plt.subplots(nrows=4,ncols=1,constrained_layout=True)
    fig1.suptitle("Lift")
    
    ax1[0].plot(t, lift)
    ax1[0].set_ylabel("Lift [mm]")
    ax1[0].set_xlim([0,360])
    
    ax1[1].plot(t, vel)
    ax1[1].set_ylabel("Vel [mm]/s")
    ax1[1].set_xlim([0,360])
    
    ax1[2].plot(t, acc)
    ax1[2].set_ylabel("Acc [mm/s^2]")
    ax1[2].set_xlabel('Theta [deg]')
    ax1[2].set_xlim([0,360])

    ax1[3].plot(t, jerk)
    ax1[3].set_ylabel("Jerk [mm/s^3]")
    ax1[3].set_xlabel('Theta [deg]')
    ax1[3].set_xlim([0,360])


# Add all four segments to the motion law:
lift, vel_deg, acc_deg, vel, acc, jerk = addMotionSegment(startangle1,endangle1,startlift1,endlift1,motionlaw1,dtheta,T)
lift, vel_deg, acc_deg, vel, acc, jerk = addMotionSegment(startangle2,endangle2,startlift2,endlift2,motionlaw2,dtheta,T)
lift, vel_deg, acc_deg, vel, acc, jerk = addMotionSegment(startangle3,endangle3,startlift3,endlift3,motionlaw3,dtheta,T)
lift, vel_deg, acc_deg, vel, acc, jerk = addMotionSegment(startangle4,endangle4,startlift4,endlift4,motionlaw4,dtheta,T)
lift, vel_deg, acc_deg, vel, acc, jerk = addMotionSegment(startangle5,endangle5,startlift5,endlift5,motionlaw5,dtheta,T)
lift, vel_deg, acc_deg, vel, acc, jerk = addMotionSegment(startangle6,endangle6,startlift6,endlift6,motionlaw6,dtheta,T)

# Plot the motion law
plotMotionLaw(theta_deg,lift,vel_deg,acc_deg, jerk_deg)

plt.show()